# 32 - Fetch v2 API results for 'software companies' and compare

Fetches 1000 results for the single query 'software companies' from the Istari v2 API (same pagination approach as `09b_api_evaluation_v2.ipynb`, just for one query instead of all 101, so quota cost is minimal: 2 pages of 500).

Then compares the fetched results against three things already on disk: the `query_id=1` ground truth pool in `dataset/company_corpus.csv`, `dataset/export.csv` (first website export), and `dataset/export (1).csv` (second website export), to see which one the live v2 API actually agrees with.

In [1]:
import os, json, time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)

ISTARI_API_KEY = os.getenv("API_KEY")
V2_BASE_URL    = "https://api.istari.ai/v2/search"
RESULT_DIR     = Path("result/32_api_v2_check")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
QUERY          = "software companies"
TARGET_SIZE    = 1000
V2_MAX_PAGE_SIZE = 500
V2_COLUMNS     = ["domain", "name", "country", "summary"]

print(f"API key set: {'yes' if ISTARI_API_KEY else 'NO -- check API_KEY in .env'}")

API key set: yes


In [2]:
# Small live quota check before spending the full 1000-result fetch.
r = requests.post(
    V2_BASE_URL,
    headers={"Accept": "application/json", "x-api-key": ISTARI_API_KEY, "Content-Type": "application/json"},
    json={"describe": QUERY, "keywords": {"must_all": [], "must_any": [], "must_not": []},
          "filters": {"country": [], "state": [], "region": [], "organization_type": [],
                      "organization_size": [], "nace_code": []},
          "excludes": [], "columns": ["domain", "name"], "size": 3},
    timeout=15,
)
if r.status_code != 200:
    raise SystemExit(f"v2 API error {r.status_code}: {r.text[:200]}")

remaining_hdr = r.headers.get("X-RateLimit-Results-Remaining")
remaining = int(remaining_hdr) if remaining_hdr is not None else None
print(f"v2 API OK. Results remaining this month: {remaining if remaining is not None else 'unlimited / not reported'}")
if remaining is not None and remaining < TARGET_SIZE:
    print(f"WARNING: only {remaining} results left, less than the {TARGET_SIZE} this fetch needs.")

v2 API OK. Results remaining this month: 3770


In [3]:
def fetch_page(size, search_after=None):
    """One page of the v2 API, paginated via the search_after cursor."""
    payload = {"describe": QUERY, "keywords": {"must_all": [], "must_any": [], "must_not": []},
               "filters": {"country": [], "state": [], "region": [], "organization_type": [],
                           "organization_size": [], "nace_code": []},
               "excludes": [], "columns": V2_COLUMNS, "size": size}
    if search_after is not None:
        payload["search_after"] = search_after
    resp = requests.post(
        V2_BASE_URL,
        headers={"Accept": "application/json", "x-api-key": ISTARI_API_KEY, "Content-Type": "application/json"},
        json=payload, timeout=30,
    )
    if resp.status_code != 200:
        raise SystemExit(f"v2 API error {resp.status_code} on page fetch: {resp.text[:200]}")
    body = resp.json()
    return body.get("data", []), body.get("metadata", {}).get("search_after")


def fetch_query(target_size):
    all_data, seen_domains, search_after = [], set(), None
    while len(all_data) < target_size:
        page_size = min(V2_MAX_PAGE_SIZE, target_size - len(all_data))
        data, next_cursor = fetch_page(page_size, search_after)
        new_rows = [row for row in data if row["domain"] not in seen_domains]
        seen_domains.update(row["domain"] for row in new_rows)
        for rank, row in enumerate(new_rows, start=len(all_data) + 1):
            row["rank"] = rank
        all_data.extend(new_rows)
        if not data or next_cursor is None:
            break
        search_after = next_cursor
        time.sleep(1.5)
    return all_data


print(f"Fetching {TARGET_SIZE} results for '{QUERY}'...")
api_results = fetch_query(TARGET_SIZE)
print(f"Got {len(api_results)} unique-domain results.")

out_path = RESULT_DIR / "software_companies_v2_1000.json"
with open(out_path, "w") as f:
    json.dump(api_results, f, indent=2, default=str)
print(f"Saved to {out_path}")

Fetching 1000 results for 'software companies'...
Got 1000 unique-domain results.
Saved to result/32_api_v2_check/software_companies_v2_1000.json


In [4]:
api_df = pd.DataFrame(api_results)

corpus = pd.read_csv("dataset/company_corpus.csv")
q1 = corpus[corpus["query_id"] == 1].reset_index(drop=True)

export1 = pd.read_csv("dataset/export.csv")
export2 = pd.read_csv("dataset/export (1).csv")


def compare(name, other_domains):
    overlap = set(api_df["domain"]) & set(other_domains)
    print(f"{name}: {len(overlap)}/{len(api_df)} of the v2 API's top-{len(api_df)} also appear in {name}")


print(f"v2 API fetched {len(api_df)} results for '{QUERY}'\n")
compare("query_id=1 ground truth pool (1000)", q1["domain"])
compare("export.csv (first website export, 100)", export1["domain"])
compare("export (1).csv (second website export, 100)", export2["domain"])

merged_gt = api_df.merge(q1[["domain", "rank"]], on="domain", how="inner", suffixes=("_api", "_gt"))
print(f"\nOf the overlapping domains with the ground truth pool, ground-truth rank stats:")
print(merged_gt["rank_gt"].describe())

v2 API fetched 1000 results for 'software companies'

query_id=1 ground truth pool (1000): 391/1000 of the v2 API's top-1000 also appear in query_id=1 ground truth pool (1000)
export.csv (first website export, 100): 28/1000 of the v2 API's top-1000 also appear in export.csv (first website export, 100)
export (1).csv (second website export, 100): 100/1000 of the v2 API's top-1000 also appear in export (1).csv (second website export, 100)

Of the overlapping domains with the ground truth pool, ground-truth rank stats:
count    391.000000
mean     370.086957
std      284.026430
min        1.000000
25%      122.500000
50%      305.000000
75%      582.000000
max      993.000000
Name: rank_gt, dtype: float64
